In [1]:
# CELDA 1: Preparación del entorno
import sys
import os

# 1. Clonamos el repositorio
!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git -b diego_branch

# 2. Instalamos las dependencias necesarias
!pip install polars scikit-learn transformers torch tqdm

# 3. Le decimos a Python dónde buscar los módulos de (src/models)
# Esto es vital para poder importar train_classic y train_transformer
sys.path.append('/content/proyecto-transformacion-texto-imagen')
sys.path.append('/content/proyecto-transformacion-texto-imagen/src/models')

print("✅ Entorno configurado correctamente.")

Cloning into 'proyecto-transformacion-texto-imagen'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (176/176), done.
remote: Total 231 (delta 101), reused 166 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (231/231), 3.08 MiB | 11.83 MiB/s, done.
Resolving deltas: 100% (101/101), done.
✅ Entorno configurado correctamente.


In [2]:
# CELDA 2: Carga de datos
import polars as pl
import numpy as np
from typing import Tuple, List

def cargar_datos() -> Tuple[List[str], np.ndarray, List[str], np.ndarray]:
    """
    Carga los archivos particionados usando Polars y extrae las listas necesarias.
    Extraemos train y validation para el Fine-Tuning.
    """
    ruta_base = '/content/proyecto-transformacion-texto-imagen/data/processed/'

    # Cargamos los datos limpios (Limpieza Básica)
    df_train = pl.read_csv(ruta_base + 'train.csv')
    df_test = pl.read_csv(ruta_base + 'test.csv') # Usaremos test para evaluar LogReg

    # Extraemos textos y etiquetas
    X_train: List[str] = df_train.get_column("text").to_list()
    y_train: np.ndarray = df_train.get_column("manual_classification").to_numpy()

    X_test: List[str] = df_test.get_column("text").to_list()
    y_test: np.ndarray = df_test.get_column("manual_classification").to_numpy()

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = cargar_datos()
print(f"✅ Datos cargados: {len(X_train)} para entrenamiento, {len(X_test)} para prueba.")

✅ Datos cargados: 908 para entrenamiento, 114 para prueba.


In [3]:
# CELDA 3: Extractor de Embeddings
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

def extraer_embeddings_frozen(textos: List[str], nombre_modelo: str, batch_size: int = 16) -> np.ndarray:
    """
    Pasa los textos por el Transformer congelado y extrae el token [CLS].

    Args:
        textos: Lista de strings a procesar.
        nombre_modelo: Identificador de HuggingFace (ej. 'xlm-roberta-base').
        batch_size: Cuántos textos procesar a la vez para no saturar la RAM de la GPU.

    Returns:
        np.ndarray: Matriz de características numéricas (Embeddings).
    """
    print(f"Cargando modelo {nombre_modelo} para extracción de características...")

    # 1. Cargamos el tokenizador y el modelo base (sin capa de clasificación)
    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    modelo = AutoModel.from_pretrained(nombre_modelo)

    # 2. Movemos el modelo a la GPU y lo ponemos en modo evaluación (congela capas como Dropout)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    modelo = modelo.to(device)
    modelo.eval()

    embeddings: List[np.ndarray] = []

    # 3. Procesamos en lotes (batches)
    for i in tqdm(range(0, len(textos), batch_size), desc="Extrayendo"):
        batch_textos = textos[i:i+batch_size]

        # Convertimos texto a tokens. max_length=128 por compatibilidad con script de Santiago
        inputs = tokenizer(batch_textos, padding=True, truncation=True, max_length=128, return_tensors="pt")
        inputs = inputs.to(device)

        # torch.no_grad() es el corazón del concepto "Frozen".
        # Apaga el cálculo de gradientes, ahorrando muchísima memoria y evitando el entrenamiento.
        with torch.no_grad():
            outputs = modelo(**inputs)
            # outputs.last_hidden_state tiene forma: [batch_size, sequence_length, hidden_size]
            # Seleccionamos [:, 0, :] para tomar TODOS los lotes, solo el PRIMER token [CLS], y TODOS sus pesos.
            cls_tokens = outputs.last_hidden_state[:, 0, :]

            # Lo pasamos de vuelta al CPU y lo convertimos a un arreglo de NumPy que scikit-learn entienda
            embeddings.append(cls_tokens.cpu().numpy())

    # Unimos todos los lotes en una sola gran matriz
    return np.vstack(embeddings)

In [4]:
# CELDA 4: Ejecución de Experimentos
# Importamos los scripts de tu equipo
from src.models.train_classic import train_logistic
from src.models.train_transformer import run_finetuning

# Nombres de los modelos en HuggingFace
MODELO_BETO = "dccuchile/bert-base-spanish-wwm-cased"
MODELO_XLM = "xlm-roberta-base"

# =====================================================================
# FAMILIA 2: FROZEN + REGRESIÓN LOGÍSTICA
# =====================================================================

print("\\n" + "="*50)
print("EXPERIMENTO 2.1: BETO (Frozen) + Regresión Logística")
# 1. Extraemos features
X_train_emb_beto = extraer_embeddings_frozen(X_train, MODELO_BETO)
X_test_emb_beto = extraer_embeddings_frozen(X_test, MODELO_BETO)
# 2. Entrenamos (usamos la función exacta de Santiago)
train_logistic(X_train_emb_beto, y_train, X_test_emb_beto, y_test, experiment_name="Exp_2.1_BETO_Frozen")


print("\\n" + "="*50)
print("EXPERIMENTO 2.4: XLM-RoBERTa (Frozen) + Regresión Logística")
X_train_emb_xlm = extraer_embeddings_frozen(X_train, MODELO_XLM)
X_test_emb_xlm = extraer_embeddings_frozen(X_test, MODELO_XLM)
train_logistic(X_train_emb_xlm, y_train, X_test_emb_xlm, y_test, experiment_name="Exp_2.4_XLM_Frozen")


# =====================================================================
# FAMILIA 3: FINE-TUNING
# =====================================================================
# OJO: Estos tomarán tiempo. El script de Santiago ya tiene configurado el Trainer.

print("\\n" + "="*50)
print("EXPERIMENTO 3.1: BETO (Fine-Tuning)")
# Nota: Pasamos X_test y y_test como set de validación para que el Trainer de Santiago
# reporte las métricas de evaluación época por época.
run_finetuning(X_train, y_train, X_test, y_test, model_name=MODELO_BETO)


print("\\n" + "="*50)
print("EXPERIMENTO 3.4: XLM-RoBERTa (Fine-Tuning)")
run_finetuning(X_train, y_train, X_test, y_test, model_name=MODELO_XLM)

print("\\n✅ TODOS LOS EXPERIMENTOS FINALIZADOS.")

\n==================================================
EXPERIMENTO 2.1: BETO (Frozen) + Regresión Logística
Cargando modelo dccuchile/bert-base-spanish-wwm-cased para extracción de características...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final Evaluation Metrics:
{'eval_loss': 0.6322817206382751, 'eval_accuracy': 0.7192982456140351, 'eval_precision_macro': 0.7051282051282051, 'eval_recall_macro': 0.6870129870129871, 'eval_f1_macro': 0.6918918918918919, 'eval_runtime': 1.0445, 'eval_samples_per_second': 109.141, 'eval_steps_per_second': 7.659, 'epoch': 6.0}

Confusion Matrix:
[[58 12]
 [20 24]]
\n✅ TODOS LOS EXPERIMENTOS FINALIZADOS.


In [5]:
# CELDA 5: Guardar resultados en el CSV
import os
import csv
from sklearn.metrics import f1_score
from src.models.train_classic import train_logistic

def guardar_resultado_csv(experimento: str, preprocessing: str, modelo: str, f1_macro: float):
    """
    Agrega una nueva fila al archivo de resultados del equipo.
    Si el archivo no existe, lo crea con los encabezados.
    """
    ruta_csv = '/content/proyecto-transformacion-texto-imagen/reports/resultados_diego_ablacion.csv'

    # Crear la carpeta reports si no existe
    os.makedirs(os.path.dirname(ruta_csv), exist_ok=True)

    archivo_existe = os.path.isfile(ruta_csv)

    with open(ruta_csv, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        # Si es la primera vez que se crea, poner encabezados
        if not archivo_existe:
            writer.writerow(['Experimento', 'Preprocessing', 'Modelo', 'Macro F1 Score'])

        writer.writerow([experimento, preprocessing, modelo, f1_macro])

    print(f"✅ Resultado guardado en CSV: {experimento} | F1: {f1_macro:.4f}")

# ==========================================
# RECOLECCIÓN DE MÉTRICAS Y GUARDADO
# ==========================================
print("\nGuardando métricas de la Familia 2...")

# 1. Calculamos las predicciones de los modelos Frozen para obtener el F1-Score exacto
from sklearn.linear_model import LogisticRegression
import joblib

# Cargamos el modelo 2.1 que guardó el script de Santiago
modelo_2_1 = joblib.load('/content/proyecto-transformacion-texto-imagen/models/Exp_2.1_BETO_Frozen.pkl')
preds_2_1 = modelo_2_1.predict(X_test_emb_beto)
f1_2_1 = f1_score(y_test, preds_2_1, average='macro')

# Guardamos en el CSV
guardar_resultado_csv("2.1 (Frozen)", "Limpieza Básica", "BETO + LogReg", f1_2_1)

# Cargamos el modelo 2.4
modelo_2_4 = joblib.load('/content/proyecto-transformacion-texto-imagen/models/Exp_2.4_XLM_Frozen.pkl')
preds_2_4 = modelo_2_4.predict(X_test_emb_xlm)
f1_2_4 = f1_score(y_test, preds_2_4, average='macro')

guardar_resultado_csv("2.4 (Frozen)", "Limpieza Básica", "XLM-RoBERTa + LogReg", f1_2_4)

print("\nNota: Para los modelos de la Familia 3 (Fine-Tuning), el Trainer de HuggingFace guarda un log detallado en la carpeta de outputs. Deberás revisar el último F1-Score reportado en consola y agregarlo al CSV manualmente o extraerlo del objeto Trainer.")


Guardando métricas de la Familia 2...
✅ Resultado guardado en CSV: 2.1 (Frozen) | F1: 0.6669
✅ Resultado guardado en CSV: 2.4 (Frozen) | F1: 0.6800

Nota: Para los modelos de la Familia 3 (Fine-Tuning), el Trainer de HuggingFace guarda un log detallado en la carpeta de outputs. Deberás revisar el último F1-Score reportado en consola y agregarlo al CSV manualmente o extraerlo del objeto Trainer.


In [6]:
import polars as pl
from typing import Dict

def agregar_resultados_finetuning(ruta_csv: str) -> None:
    """
    Agrega los resultados manuales del Fine-Tuning al CSV usando Polars.
    """
    # Resultados extraídos de los logs
    nuevos_datos: Dict[str, list] = {
        "Experimento": ["3.1 (Fine-Tuning)", "3.4 (Fine-Tuning)"],
        "Preprocessing": ["Limpieza Básica", "Limpieza Básica"],
        "Modelo": ["Transformer (BETO)", "Transformer (XLM-RoBERTa)"],
        "Macro F1 Score": [0.741132, 0.691891]
    }

    df_nuevos = pl.DataFrame(nuevos_datos)

    # Leemos el archivo actual
    df_actual = pl.read_csv(ruta_csv)

    # Concatenamos verticalmente
    df_final = pl.concat([df_actual, df_nuevos])

    # Guardamos sobrescribiendo el archivo
    df_final.write_csv(ruta_csv)
    print("✅ Resultados de Fine-Tuning agregados exitosamente con Polars.")
    print(df_final)

# Ejecutamos
RUTA = '/content/proyecto-transformacion-texto-imagen/reports/resultados_diego_ablacion.csv'
agregar_resultados_finetuning(RUTA)

✅ Resultados de Fine-Tuning agregados exitosamente con Polars.
shape: (7, 4)
┌─────────────────────┬──────────────────────────┬───────────────────────────┬────────────────┐
│ Experimento         ┆ Preprocessing            ┆ Modelo                    ┆ Macro F1 Score │
│ ---                 ┆ ---                      ┆ ---                       ┆ ---            │
│ str                 ┆ str                      ┆ str                       ┆ f64            │
╞═════════════════════╪══════════════════════════╪═══════════════════════════╪════════════════╡
│ 1.2 (P3+TFIDF)      ┆ Lematización + StopWords ┆ LogReg                    ┆ 0.669565       │
│ 2.2 (P2+Embeddings) ┆ StopWords Custom         ┆ LogReg                    ┆ 0.752422       │
│ 3.2 (P2+FineTuning) ┆ StopWords Custom         ┆ Transformer (Beto)        ┆ 0.719948       │
│ 2.1 (Frozen)        ┆ Limpieza Básica          ┆ BETO + LogReg             ┆ 0.666883       │
│ 2.4 (Frozen)        ┆ Limpieza Básica          ┆ XLM-RoBE